#  **Data Collection and Preprocessing**

### Import Libraries

In [1]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, parent_dir)

In [2]:
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
)
from src.data_scraping import scrap_reviews

### Web Scraping

#### App metadata

In [3]:
CBE_APP_ID = 'com.combanketh.mobilebanking'
display_app_info(CBE_APP_ID)

Commercial Bank of Ethiopia App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.2835107
Total Ratings: 48,537
Total Reviews: 9,329
Installs     : 10,000,000+


#### Scrape reviews

In [4]:
reviews = scrap_reviews(app_id=CBE_APP_ID, num_reviews=1000)

Scraping reviews for com.combanketh.mobilebanking...
Collected 1000 raw reviews


#### Collect review text, rating, review date, bank , source

In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 9f0c0b92-5087-496f-98e2-60a97f2076d1
  userName: Tesfae Endeshaw
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocI2cU7JkP-cEzHioUEV_SFWIKrjkzH-P4nLfnbeClq7xGhB2g=mo
  content: very nice
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-17 14:33:09
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [6]:
df = review_dataframe(reviews, app_info={'title': 'CBE Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (1000, 6)


,review_id,review,rating,date,bank,source
0,9f0c0b92-5087-496f-98e2-60a97f2076d1,very nice,5,2026-05-17 14:33:09,CBE Bank,Google Play
1,7b8203d4-e95a-4693-b2ea-3e35a6aa797d,f*k,1,2026-05-17 13:23:30,CBE Bank,Google Play
2,60e3a921-2d6b-44a8-bb4b-636af4637b83,The Bank You can Always Rely on!,5,2026-05-17 13:10:51,CBE Bank,Google Play
3,92e988a8-485c-4691-90ff-4005e0f0b3b7,Please make the CBE Noor toggle to be optional...,2,2026-05-17 09:25:49,CBE Bank,Google Play
4,d7c8285b-d811-4738-ba6e-98014135b5a3,Amazing App,5,2026-05-17 04:11:07,CBE Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [7]:
df_clean = df.copy()

In [8]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 1000 reviews


#### Handle missing values

In [9]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 1000 reviews


#### Normalize dates to YYYY-MM-DD format

In [10]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-17 14:33:09
1   2026-05-17 13:23:30
2   2026-05-17 13:10:51
dtype: datetime64[us]

After normalization:
0    2026-05-17
1    2026-05-17
2    2026-05-17
dtype: str

Date range: 2025-12-16 to 2026-05-17


#### Handle incorrect ratings

In [11]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 1000 reviews


#### Clean review text

In [12]:
df_clean['review'] = df_clean['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df_clean['review'].head(10).to_string())

Sample cleaned reviews:
0                                            very nice
1                                                  f*k
2                     the bank you can always rely on!
3    please make the cbe noor toggle to be optional...
4                                          amazing app
5    it stopped working on its own. when you check ...
6    the most backward and unstable financial app i...
7                                                   ok
8                                                 good
9                                                     


#### Save the cleaned dataset

In [13]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']]

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (1000, 5)


,review,rating,date,bank,source
0,very nice,5,2026-05-17,CBE Bank,Google Play
1,the bank you can always rely on!,5,2026-05-17,CBE Bank,Google Play
2,please make the cbe noor toggle to be optional...,2,2026-05-17,CBE Bank,Google Play
3,amazing app,5,2026-05-17,CBE Bank,Google Play
4,f*k,1,2026-05-17,CBE Bank,Google Play
5,it stopped working on its own. when you check ...,1,2026-05-16,CBE Bank,Google Play
6,the most backward and unstable financial app i...,1,2026-05-16,CBE Bank,Google Play
7,ok,5,2026-05-16,CBE Bank,Google Play
8,good,5,2026-05-16,CBE Bank,Google Play
9,,5,2026-05-16,CBE Bank,Google Play


In [15]:
save_cleaned_data(df_clean, output_path="../../data/processed/cbe_reviews_cleaned.csv")

Cleaned data saved to ../../data/processed/cbe_reviews_cleaned.csv
Saved to: ../../data/processed/cbe_reviews_cleaned.csv


### Report

In [16]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :   1000
  Reviews after cleaning :   1000
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-12-16  to  2026-05-17
Rating distribution:
  5 stars:  661  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:   82  ████████████████
  3 stars:   62  ████████████
  2 stars:   45  █████████
  1 stars:  150  ██████████████████████████████

  Text length stats:
    Min    : 0 characters
    Median : 13 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

